# LogiScan Phase 4: Unified Classifier Training

This notebook trains the 24-class fine-grained fallacy classifier for LogiScan Phase 4 (Boundary Refinement).

### Setup
1. Upload `unified_training_data.json` to the file explorer on the left.
2. Ensure you are using a GPU runtime (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!pip install -q transformers[torch] datasets accelerate scikit-learn tqdm

In [ ]:
import json

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

### Load and Prepare Data

In [ ]:
DATA_PATH = "unified_training_data.json"

with open(DATA_PATH) as f:
    data = json.load(f)

texts = [d["text"] for d in data]
label_list = sorted(list(set(d["fallacy"] for d in data)))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
labels = [label2id[d["fallacy"]] for d in data]

# Calculate class weights
label_counts = np.bincount(labels)
weights = 1.0 / (label_counts + 1e-6)
weights = weights / weights.sum() * len(label_list)
class_weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)

print(f"Loaded {len(texts)} samples across {len(label_list)} classes.")

X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.15, random_state=42, stratify=labels
)

In [ ]:
class FallacyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

### Training Initialization

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-small" # Better performance if sentencepiece is available
try:
    import sentencepiece
except ImportError:
    !pip install -q sentencepiece

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
).to(DEVICE)

train_ds = FallacyDataset(X_train, y_train, tokenizer)
val_ds = FallacyDataset(X_val, y_val, tokenizer)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

EPOCHS = 4
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)

### Training Loop

In [ ]:
best_f1 = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch in progress:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels_batch = batch["label"].to(DEVICE)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        loss = loss_fn(logits, labels_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
        progress.set_postfix({"loss": f"{loss.item():.4f}"})

    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(
                batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE)
            )
            preds = torch.argmax(outputs.logits, dim=1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(batch["label"].numpy())

    f1 = f1_score(all_labels, all_preds, average="macro")
    print(f"\nEpoch {epoch+1} Results:")
    print(f"Val Macro-F1: {f1:.4f}")
    print(classification_report(all_labels, all_preds, target_names=label_list))

    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained("stage3_fallacy_model")
        tokenizer.save_pretrained("stage3_fallacy_model")
        with open("stage3_fallacy_model/id2label.json", "w") as f:
            json.dump(id2label, f)
        print("✅ Saved best model!")

### Download Model
Run this to zip and download the trained model.

In [ ]:
!zip -r stage3_fallacy_model.zip stage3_fallacy_model
from google.colab import files

files.download("stage3_fallacy_model.zip")